In [ ]:
!docker run -it --rm -v $(pwd)/discord_output:/app/output tyrrrz/discordchatexporter:latest exportguild -t <> -g <> --include-threads -f Json -o /app/output/

In [ ]:
import os

# Directory where the files are being generated
output_dir = 'discord_output'

# List all files in the directory
files = os.listdir(output_dir)

# Print out the file names
for file in files:
    print(file)

In [ ]:
import os
import re

def rename_discord_output_files(directory):
    """
    Renames all JSON files in the specified directory according to the format:
    'UBCStudy1_channelX_UniqueID.json', where X is the channel number and UniqueID
    is the unique identifier extracted from the original filename.

    Args:
        directory (str): The path to the directory containing the JSON files.

    Returns:
        list: A list of tuples containing the old and new file names for verification.
    """
    # Regex to extract channel number and unique ID
    channel_regex = re.compile(r'channel-(\d+)')
    id_regex = re.compile(r'\[(\d+)\]')

    renamed_files = []

    # List all files in the directory
    files = os.listdir(directory)

    # Rename each file
    for file in files:
        if file.endswith('.json'):
            # Extract channel number using regex
            channel_match = channel_regex.search(file)
            if channel_match:
                channel_number = channel_match.group(1)

            # Extract unique ID using regex
            id_match = id_regex.search(file)
            if id_match:
                unique_id = id_match.group(1)

            # Create new file name in the format: 'UBCStudy1_channelX_UniqueID.json'
            new_file_name = f"UBCStudy1_channel{channel_number}_{unique_id}.json"

            # Old and new file paths
            old_file_path = os.path.join(directory, file)
            new_file_path = os.path.join(directory, new_file_name)

            # Rename the file
            os.rename(old_file_path, new_file_path)

            # Add to the list for verification
            renamed_files.append((file, new_file_name))

    return renamed_files


In [ ]:
rename_discord_output_files('discord_output')

In [ ]:
import os

# Directory where the files are being generated
output_dir = 'discord_output'

# List all files in the directory
files = os.listdir(output_dir)

# Print out the file names
for file in files:
    print(file)

In [ ]:
import os
import json
import csv
from datetime import datetime

def parse_json_files_to_csv(directory, output_csv):
    """
    Parses all JSON files in the specified directory and writes the relevant data into a master CSV.

    Args:
        directory (str): Path to the directory containing JSON files.
        output_csv (str): Path to the output CSV file.
    """
    # Prepare the CSV headers
    csv_headers = [
        'guild_name', 'channel_name', 'channel_type', 'message_timestamp',
        'participant_name', 'message_id', 'message_content'
    ]

    # Open the CSV file for writing
    with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=csv_headers)
        writer.writeheader()

        # Iterate through all JSON files in the directory
        for file in os.listdir(directory):
            if file.endswith('.json'):
                file_path = os.path.join(directory, file)

                # Parse the JSON file
                with open(file_path, 'r') as f:
                    data = json.load(f)

                    guild_name = data["guild"]["name"]
                    channel_name = data["channel"]["name"]
                    channel_type = data["channel"]["type"]

                    # Extract messages
                    for message in data["messages"]:
                        participant_name = message["author"]["name"]
                        message_id = message["id"]
                        message_timestamp = message["timestamp"]
                        message_content = message.get("content", "")

                        # Write the extracted data into the CSV
                        writer.writerow({
                            'guild_name': guild_name,
                            'channel_name': channel_name,
                            'channel_type': channel_type,
                            'message_timestamp': message_timestamp,
                            'participant_name': participant_name,
                            'message_id': message_id,
                            'message_content': message_content
                        })

# Example usage:
directory = 'discord_output'
output_csv = 'discord_output/master_discord_data.csv'
parse_json_files_to_csv(directory, output_csv)


In [ ]:
import pandas as pd
from datetime import datetime, timedelta

def filter_inactive_participants(input_csv_path, output_csv_path):
    # Load the CSV data
    df = pd.read_csv(input_csv_path)
    
    # Parse the 'message_timestamp' into datetime format and make it timezone-aware (UTC)
    df['message_timestamp'] = pd.to_datetime(df['message_timestamp'], utc=True, errors='coerce')
    
    # Define the 24-hour threshold using timezone-aware UTC datetime
    current_time = datetime.utcnow().replace(tzinfo=pd.Timestamp.utcnow().tzinfo)  # Assuming UTC time for comparison
    time_threshold = current_time - timedelta(hours=24)
    
    # Find the latest engagement of each participant per channel
    last_engagement = df.groupby(['channel_name', 'participant_name'], as_index=False).agg({
        'message_timestamp': 'max'
    })
    
    # Filter participants who have not engaged in the last 24 hours
    inactive_participants = last_engagement[last_engagement['message_timestamp'] < time_threshold]
    
    # Save the results to a new CSV
    inactive_participants.to_csv(output_csv_path, index=False)
    
    print(f"Filtered inactive participants have been saved to {output_csv_path}")

# Example usage
filter_inactive_participants('discord_output/master_discord_data.csv', 'discord_output/inactive_participants.csv')
